# Soft interval overlap for collision losses

In gradient-based layout optimization, collision between shapes can be penalized by measuring how much their bounding intervals overlap. The overlap between two 1D intervals is `min(r1, r2) - max(l1, l2)`, clamped to zero.

For the loss to drive gradient descent, it must be differentiable everywhere — but `min`, `max`, and `relu` all have zero or undefined gradients away from the transition. Replacing them with smooth approximations fixes this:

- **Softplus** smooths the final `relu` clamp: `log(1 + exp(x))`
- **LogSumExp** smooths the inner `min`/`max`: `soft_max(a, b) = log(exp(βa) + exp(βb)) / β`

The soft `min`/`max` also solves a subtler problem: when one interval is entirely contained inside the other, the hard overlap equals the inner interval's length — a constant with respect to position, so its gradient is zero. The optimizer receives no signal about which direction to move the shapes apart. The soft versions remain sensitive to position even in this fully-contained case, providing a gradient that pushes the shapes toward separation.

A sharpness parameter **β** controls the trade-off: larger β gives a tighter approximation of the hard overlap (stronger signal only when shapes actually collide), while smaller β creates a longer-range repulsive force that pushes shapes apart even before they touch.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

In [ ]:
interval_1 = np.array([0, 1])
interval_2 = np.array([0, 0.5])

def calculate_hard_overlap(interval_1, interval_2):
    hard_overlap_before_relu = min(interval_1[1], interval_2[1]) - max(interval_1[0], interval_2[0])
    hard_overlap = max(0, hard_overlap_before_relu)
    return hard_overlap

def calculate_softplus_overlap(interval_1, interval_2):
    overlap_before_relu = min(interval_1[1], interval_2[1]) - max(interval_1[0], interval_2[0])
    soft_overlap = np.log(1 + np.exp(overlap_before_relu))
    return soft_overlap

beta_clamp = 3.0  # sharpness of the softplus zero-clamp

_, ax = plt.subplots(figsize=(4, 3))
ax.plot(interval_1, [0, 0], "k", marker=".")
ax.plot(interval_2, [1, 1], "r", marker=".")

dx_list = np.linspace(-1, 2, 100)
shifted = interval_2 + dx_list[:, None]  # shape (100, 2)
raw_overlap = np.minimum(interval_1[1], shifted[:, 1]) - np.maximum(interval_1[0], shifted[:, 0])

hard_overlap_list = np.maximum(0, raw_overlap)
soft_overlap_list = np.log(1 + np.exp(beta_clamp * raw_overlap)) / beta_clamp

_, ax = plt.subplots(figsize=(4, 3))
ax.plot(dx_list, hard_overlap_list, label="hard overlap")
ax.plot(dx_list, soft_overlap_list, label="soft overlap")

In [ ]:
dx_list = np.linspace(-1, 2, 100)
shifted = interval_2 + dx_list[:, None]  # shape (100, 2)

beta_minmax = 5.0  # sharpness of the soft min/max boundary detection
soft_max = lambda a, b: np.log(np.exp(beta_minmax * a) + np.exp(beta_minmax * b)) / beta_minmax
soft_min = lambda a, b: -np.log(np.exp(-beta_minmax * a) + np.exp(-beta_minmax * b)) / beta_minmax
raw_overlap = soft_min(interval_1[1], shifted[:, 1]) - soft_max(interval_1[0], shifted[:, 0])

hard_overlap_list = np.maximum(0, raw_overlap)

beta_clamp = 3.0  # sharpness of the softplus zero-clamp
soft_overlap_list = np.log(1 + np.exp(beta_clamp*raw_overlap))/beta_clamp

_, ax = plt.subplots(figsize=(4, 3))
ax.plot(dx_list, hard_overlap_list, label="hard overlap")
ax.plot(dx_list, soft_overlap_list, label="soft overlap")